In [1]:
# Re-define device if running in a new cell
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device re-established: {device}")

Device re-established: cuda


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy


In [12]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split # <--- THESE ARE CRUCIAL
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix 
# ... (any other imports needed like time, math, etc.)

In [13]:
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32

# --- Tokenizer and IDs (Essential for Text Processing) ---
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    # Fallback if local path is not accessible
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Granger Causality Matrix Creation (Needed for the graph) ---
def create_granger_causality_matrix(eeg_batch):
    # This function is used to create the static graph adjacency matrix
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            try:
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# --- Dataset Definition (Fixes NameError: 'EEGMetaTextH5Dataset') ---
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

# --- Collate Function Definition (Fixes DataLoader dependency) ---
def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# --- Create Dataset and Loaders (Fixes NameError: 'train_loader') ---
print("Creating Dataset and DataLoaders...")
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
TRAIN_PCT, VAL_PCT = 0.8, 0.1
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

print(f"Data loaders created: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}")

# --- Create Granger Matrix (Essential Static Graph) ---
print(f"Creating static Granger Causality matrix on {device}...")
eeg_b, _, _ = next(iter(train_loader))
granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
num_channels = eeg_b.shape[1]

# Handle empty graphs and add self-loops
if granger_edge_index.numel() == 0:
    print("Warning: Generated Granger matrix is empty. Creating a fallback graph with self-loops.")
    granger_edge_index = torch.arange(num_channels, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    granger_edge_attr = torch.ones(num_channels, dtype=torch.float)

granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)

# Move to device and ensure correct types
granger_edge_index = granger_edge_index.to(torch.long).to(device)
granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
print("Granger Matrix created, loaded to device, and ready for Transformer training. 🧠")

Creating Dataset and DataLoaders...
Data loaders created: Train=22400, Val=2800, Test=2800
Creating static Granger Causality matrix on cuda...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger Matrix created, loaded to device, and ready for Transformer training. 🧠


In [15]:
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    """Injects positional information into the sequence."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x is (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1) # For multi-head expansion
        
        batch_size = query.size(0)
        
        # 1) Linear projections and split into heads
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        # 2) Apply Scaled Dot Product Attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        # 3) Concatenate heads and final linear layer
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # 1. Self-Attention Sub-layer
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        # 2. Feed-Forward Sub-layer
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Gating/Conditional Bias Layer for Metadata Fusion (Option B)
        self.meta_gate = nn.Sequential(
            nn.Linear(d_meta, d_model),
            nn.Sigmoid()
        )
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask, meta_features):
        # 1. Masked Self-Attention
        # x is (B, T_tgt, D), memory (H_enc_tf) is (B, T_src, D)
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        
        # 2. Cross-Attention (Multimodal Fusion - EEG)
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        
        # 3. Metadata Conditional Bias (Modality Fusion - Metadata)
        # Apply gating/bias to the output of the cross-attention
        # meta_features is (B, D_meta) -> meta_gate output is (B, D_model)
        gate = self.meta_gate(meta_features).unsqueeze(1) # (B, 1, D_model)
        x = x * gate # Broadcast the gate across the sequence length (T_tgt)
        
        # 4. Feed-Forward
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x


In [16]:
# --- 2. New SpatioTemporalEEGEncoder (GCN + Transformer Encoder) ---
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=256, num_layers=2, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        # 1. Spatial Processing (GCN) - Same as existing
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.spatial_dropout = nn.Dropout(dropout)

        # 2. Positional Encoding
        self.pos_encoding = PositionalEncoding(d_model)

        # 3. Temporal Processing (Transformer Encoder Stack)
        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        # 4. CLS Token for MetaHead prediction
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # --- Spatial Processing (GCN) ---
        batch_edge_index, batch_edge_attr = self._prepare_gcn_input(batch_size, num_timesteps, edge_index, edge_attr)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels) # (B*T, C)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.spatial_dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        
        # Reshape to (B, T, D_model)
        x = x.reshape(batch_size, num_timesteps, self.d_model)
        
        # --- Transformer Input Preparation ---
        # 1. Prepend CLS token
        cls_token = self.cls_token.repeat(batch_size, 1, 1) # (B, 1, D_model)
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        # 2. Add Positional Encoding (Note: CLS token gets position 0)
        x = self.pos_encoding(x)

        # --- Temporal Processing (Transformer Encoder Stack) ---
        for layer in self.transformer_layers:
            x = layer(x)
        
        # H_enc_tf is the final output of the Transformer Encoder
        return self.layer_norm(x)

    def _prepare_gcn_input(self, batch_size, num_timesteps, edge_index, edge_attr):
        """Helper to tile the graph for the batch processing."""
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=edge_index.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        return batch_edge_index, batch_edge_attr


In [17]:
# --- 3. New Decoder (Transformer Decoder) ---
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, emb_dim, d_model, num_layers, num_heads, d_ff, pad_id, dropout, d_meta):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(emb_dim)
        
        # Linear layer to match embedding dim to d_model for the transformer stack
        self.input_projection = nn.Linear(emb_dim, d_model) 
        
        decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, d_meta)
        self.transformer_layers = get_clones(decoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, target_text_ids, memory, memory_mask, meta_features):
        # 1. Embedding and Positional Encoding
        tgt_embed = self.embedding(target_text_ids) # (B, T_tgt) -> (B, T_tgt, Emb_dim)
        x = self.pos_encoding(tgt_embed)
        
        # 2. Project to d_model dimension
        x = self.input_projection(x) # (B, T_tgt, D_model)

        # 3. Create causal (look-ahead) mask
        tgt_seq_len = target_text_ids.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(x.device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0) # (1, 1, T, T)
        
        # 4. Transformer Decoder Stack
        for layer in self.transformer_layers:
            # memory is the encoder output (H_enc_tf)
            x = layer(x, memory, memory_mask, tgt_mask, meta_features)
        
        x = self.layer_norm(x)
        return self.fc_out(x)


In [22]:
import torch.nn as nn

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim


    def forward(self, metadata):

        # Slice the tensor into its components
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:]
        
        # Ensure the input to the linear layer is a float tensor
        object_features_raw = object_features_raw.float()

        # Get embeddings for categorical features
        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        
        # Process the multi-hot object vector through the MLP
        object_vec = self.object_processor(object_features_raw)

        # Concatenate all features into a single vector
        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        
        return combined_features

In [23]:
# --- 4. New Seq2Seq Model (Transformer backbone) ---
class Seq2SeqTF(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, 
                 d_model=256, num_layers=4, num_heads=8, d_ff=1024, pad_id=0, dropout=0.1, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        
        # Note: d_model is used as both GCN output dim and Transformer block dim
        self.encoder = SpatioTemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.meta_encoder = MetadataEncoder(
            num_colors, num_categories, num_objects, color_emb_dim, category_emb_dim, object_feature_dim
        )
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = DecoderTF(
            text_vocab_size, d_model, d_model, num_layers, num_heads, d_ff, pad_id, dropout, meta_features_dim
        )
        
        # The input to the meta_head is the CLS token feature from the encoder
        self.meta_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects
        self.pad_id = pad_id

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr):
        # 1. Encode EEG (EEG_features is H_enc_tf, includes CLS token at pos 0)
        eeg_features = self.encoder(eeg, edge_index, edge_attr) # (B, T+1, D_model)
        
        # 2. Encode Metadata
        meta_features = self.meta_encoder(metadata) # (B, D_meta)
        
        # 3. Predict metadata using the CLS token feature
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        meta_preds = self.meta_head(cls_token_feature)
        
        # 4. Decode Text
        # Decoder memory is the EEG features WITHOUT the CLS token
        decoder_memory = eeg_features[:, 1:, :] # (B, T, D_model)
        
        # No source mask (since there's no padding in the EEG sequence)
        memory_mask = None 
        
        # Shift target_text_ids right (remove EOS, use as input)
        text_logits = self.decoder(
            target_text[:, :-1], 
            decoder_memory, 
            memory_mask, 
            meta_features
        )
        
        # Slice predictions for each metadata component
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]
        
        # text_logits is (B, T_tgt-1, V), which aligns with target_text[:, 1:]
        return text_logits, pred_color, pred_category, pred_object

# --- Model Instantiation and Training (using existing setup) ---

# Re-instantiate the model with the new class
# Parameters are chosen to maintain a similar scale to your original RNN model (d_model=256, num_layers=4)
new_model = Seq2SeqTF(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_categories=NUM_CATEGORIES,
    num_objects=NUM_OBJECTS,
    d_model=256,
    num_layers=4,
    num_heads=8,
    d_ff=1024,
    pad_id=PAD_ID,
    dropout=0.1
).to(device)


NameError: name 'GCNConv' is not defined

In [22]:
# New Training Loop functions to match the simplified forward pass
def train_one_epoch_tf(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training TF", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        # Forward pass is simplified: no teacher forcing ratio needed, as target_text[:, :-1] is always the input
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        # Target is shifted left: txt_b[:, 1:]
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_tf(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        # During evaluation, we still pass the ground truth text as input
        # for a loss calculation, but for *generation*, a separate function is needed.
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        total_loss += loss.item()
        
    return total_loss / len(loader)

# --- Training Loop (to be executed in your environment) ---
# Note: You need to re-initialize your optimizer and scheduler with new_model.
# new_optimizer = AdamW(new_model.parameters(), lr=3e-5, weight_decay=1e-2)
# new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)

# print(f"New Transformer Model instantiated on '{device}'.")
# print(f"Total parameters: {sum(p.numel() for p in new_model.parameters() if p.requires_grad):,}")

# EPOCHS = 20
# best_val_loss = float('inf')
# print("\n--- Starting Transformer Training ---")
# for epoch in range(1, EPOCHS + 1):
#     start_time = time.time()
#     train_loss = train_one_epoch_tf(new_model, train_loader, new_optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
#     val_loss = evaluate_tf(new_model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    
#     new_scheduler.step(val_loss)
#     # ... (rest of the print and save logic)

In [26]:
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH) 
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [n_train, n_val, n_test], generator=g)

# The collate_multimodal_batch function definition must be available
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

print("Data loaders (train_loader, val_loader, test_loader) are now defined.")

Data loaders (train_loader, val_loader, test_loader) are now defined.


In [28]:
text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
object_criterion = nn.BCEWithLogitsLoss()

print("All loss criteria (text_criterion, color_criterion, etc.) are now defined.")

All loss criteria (text_criterion, color_criterion, etc.) are now defined.


In [29]:
import time
import math
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# --- 1. Initialization ---
# Initialize optimizer and scheduler for the NEW model
new_optimizer = AdamW(new_model.parameters(), lr=3e-5, weight_decay=1e-2)
new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)

# Constants
EPOCHS = 50
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model.pt'

# Print initial model stats
print(f"New Transformer Model instantiated on '{device}'.")
print(f"Total parameters: {sum(p.numel() for p in new_model.parameters() if p.requires_grad):,}")

best_val_loss = float('inf')

# --- 2. Training Loop ---
print("\n--- Starting Transformer Training ---")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # Train step
    train_loss = train_one_epoch_tf(
        new_model, train_loader, new_optimizer, 
        text_criterion, color_criterion, category_criterion, object_criterion, 
        granger_edge_index, granger_edge_attr
    )
    
    # Evaluate step
    val_loss = evaluate_tf(
        new_model, val_loader, 
        text_criterion, color_criterion, category_criterion, object_criterion, 
        granger_edge_index, granger_edge_attr
    )
    
    # Learning rate scheduler step
    new_scheduler.step(val_loss)
    
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    
    # Print results
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    # Use the loss as an upper bound for perplexity
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    
    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(new_model.state_dict(), MODEL_SAVE_PATH)
        print("\t-> Validation loss improved, saving new best Transformer model. 🏆")

print("\n--- Transformer Training Complete ---")

New Transformer Model instantiated on 'cuda'.
Total parameters: 23,527,913

--- Starting Transformer Training ---


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 01/50 | Time: 02m 48s
	Train Loss: 5.8864
	 Val. Loss: 4.1834 | Val. Perplexity: 65.5904
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 02/50 | Time: 02m 37s
	Train Loss: 3.7940
	 Val. Loss: 3.4002 | Val. Perplexity: 29.9704
	-> Validation loss improved, saving new best Transformer model. 🏆


...


Epoch 11/50 | Time: 02m 38s
	Train Loss: 1.5297
	 Val. Loss: 1.4136 | Val. Perplexity:  4.1105
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 12/50 | Time: 02m 38s
	Train Loss: 1.4267
	 Val. Loss: 1.3082 | Val. Perplexity:  3.6995
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [31]:
import torch
import torch.nn.functional as F
import statistics
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import time
import math

# --- 1. Load the Best Model Weights ---
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model.pt'
try:
    # new_model must be defined before this block
    new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
except FileNotFoundError:
    print(f"ERROR: Model file not found at {MODEL_SAVE_PATH}. Cannot run final test.")
    # You may choose to raise an error or continue with un-trained weights
    # raise

# --- 2. Inference (Text Generation) Function ---
@torch.no_grad()
# def generate_text_tf(model, eeg_signal, meta_signal, edge_index, edge_attr, max_len=64):
#     """Generates text from EEG and Metadata using greedy decoding."""
#     model.eval()
    
#     # Unsqueeze to add batch dimension (B=1) and move to device
#     eeg_signal = eeg_signal.unsqueeze(0).to(device)
#     meta_signal = meta_signal.unsqueeze(0).to(device)
    
#     # Encoder Forward Pass
#     eeg_features = model.encoder(eeg_signal, edge_index, edge_attr) # (1, T+1, D_model)
#     meta_features = model.meta_encoder(meta_signal) # (1, D_meta)
    
#     decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
#     memory_mask = None 

#     # Start with the SOS token
#     input_ids = torch.tensor([[SOS_ID]], dtype=torch.long, device=device)
#     generated_ids = []

#     # Decoding Loop (Greedy Search)
#     for _ in range(max_len):
#         logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
        
#         # Get the logits for the last token
#         next_token_logits = logits[:, -1, :] 
        
#         # Greedy choice
#         next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(0) # (1, 1)
#         token_id = next_token_id.item()
        
#         if token_id == EOS_ID:
#             break
            
#         if token_id != PAD_ID:
#             generated_ids.append(token_id)
            
#         # Append the new token to the input sequence for the next step
#         input_ids = torch.cat([input_ids, next_token_id.long()], dim=1)

#     predicted_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
#     return predicted_text

@torch.no_grad()
def generate_text_beam(model, eeg_signal, meta_signal, edge_index, edge_attr, 
                       beam_width=5, max_len=64):
    """
    Generates text using Beam Search for the Transformer Decoder.
    Note: Since the Transformer decoder processes the entire input sequence at once,
    this implementation feeds the growing input_ids to the decoder at each step.
    """
    model.eval()
    
    # Unsqueeze input to create a batch size of 1
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG and Metadata (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
    memory_mask = None 

    # 2. Initialization: Start with a list of beams
    # Each beam is: (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 3. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        # Iterate over all current beams
        for seq, score in beams:
            # Skip beams that are already finished
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            # Input sequence for the decoder (e.g., [SOS, w1, w2])
            input_ids = torch.tensor([seq], dtype=torch.long, device=device) # (1, current_len)

            # Decoder forward pass: Gets logits for all tokens up to current_len
            # Output logits is (1, current_len, vocab_size)
            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            
            # We only care about the logits for the last token in the sequence
            next_token_logits = logits[:, -1, :].squeeze(0) # (vocab_size)
            
            # Convert to log probabilities and find the top K (beam_width) candidates
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            # Create new candidate beams
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob # Accumulate log probability
                
                new_beams.append((new_seq, new_score))

        # Select the top 'beam_width' beams from the candidates
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Stop condition: If the top beam is finished
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 4. Final Output Selection
    best_seq = beams[0][0]
    
    # Remove SOS (pos 0) and EOS (pos -1, if present) tokens
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    return predicted_text
# --- 3. Loss Evaluation Function (Re-used/Modified evaluate_tf) ---
@torch.no_grad()
def test_evaluation(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    """Evaluates the model on the test set using the combined loss function."""
    model.eval()
    total_loss = 0.0
    
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        text_logits, pred_color, pred_category, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr
        )
        
        # Calculate individual losses
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        # Combined weighted loss 
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        total_loss += loss.item()
        
    return total_loss / len(loader)

# --- 4. ROUGE and Qualitative Evaluation Loop ---

# Configuration
NUM_SAMPLES_TO_PRINT = 10 
samples_printed = 0
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_scores_1_f = []
rouge_scores_2_f = []
rouge_scores_L_f = []
global_sample_index = 0

print("\n--- Starting ROUGE Evaluation and Qualitative Analysis ---")

new_model.eval()
test_progress_bar = tqdm(test_loader, desc="ROUGE Evaluation", leave=True)

start_time = time.time()
total_test_loss = 0.0

for eeg_b, meta_b, txt_b in test_progress_bar:
    
    # Calculate the combined loss for the batch (uses test_evaluation logic implicitly)
    batch_loss = test_evaluation(new_model, [(eeg_b, meta_b, txt_b)], 
                                 text_criterion, color_criterion, category_criterion, object_criterion, 
                                 granger_edge_index, granger_edge_attr)
    total_test_loss += batch_loss * len(eeg_b) # Accumulate weighted by batch size

    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i]
        true_text_ids = txt_b[i]
        
        # --- Generation ---
        predicted_text = generate_text_tf(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
        )
        
        # --- Ground Truth Text ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
        if not true_text:
            continue
            
        # --- Print Sample Analysis ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            print(f"\n--- Sample {global_sample_index + 1} ---")
            print(f"GROUND TRUTH: {true_text}")
            print(f"PREDICTION:   {predicted_text}")
            samples_printed += 1

        # --- Calculate ROUGE (FIXED LINE) ---
        # The 'rouge-score' library typically uses 'target' and 'prediction'
        scores = scorer.score(
            target=true_text,  # Corrected argument name
            prediction=predicted_text # Corrected argument name
        )
        
        rouge_scores_1_f.append(scores['rouge1'].fmeasure)
        rouge_scores_2_f.append(scores['rouge2'].fmeasure)
        rouge_scores_L_f.append(scores['rougeL'].fmeasure)

        # Update progress bar
        avg_rouge_l = statistics.mean(rouge_scores_L_f) if rouge_scores_L_f else 0
        test_progress_bar.set_postfix(avg_rouge_L=f"{avg_rouge_l:.4f}")
        
        global_sample_index += 1

# --- 5. Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

final_test_loss = total_test_loss / global_sample_index # Average over all samples
final_rouge_1 = statistics.mean(rouge_scores_1_f)
final_rouge_2 = statistics.mean(rouge_scores_2_f)
final_rouge_L = statistics.mean(rouge_scores_L_f)

print("\n=============================================")
print(f"Test Set Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Final Combined Test Loss: {final_test_loss:.4f} |")
print(f"| Final Test Perplexity:    {math.exp(final_test_loss):7.4f} |")
print("---------------------------------------------")
print(f"| ROUGE-1 (Unigram Overlap): {final_rouge_1:.4f} |")
print(f"| ROUGE-2 (Bigram Overlap):  {final_rouge_2:.4f} |")
print(f"| ROUGE-L (LCS Overlap):     {final_rouge_L:.4f} |")
print("=============================================")

Successfully loaded best model weights from: eeg-meta-text-spatiotemporal-transformer-model.pt

--- Starting ROUGE Evaluation and Qualitative Analysis ---


ROUGE Evaluation:   0%|          | 0/88 [00:00<?, ?it/s]


--- Sample 1 ---
GROUND TRUTH: a school of orange fish swims around a vibrant coral reef.. tone : serene
PREDICTION:   a vibrant orange and white clownfish swim near a vibrant coral reef... tone : serene

--- Sample 2 ---
GROUND TRUTH: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : awe - inspiring
PREDICTION:   a powerful

--- Sample 3 ---
GROUND TRUTH: vibrant fireworks explode in the night sky.. tone : energetic
PREDICTION:   a vibrant fireworks explode in the night sky... tone : exciting

--- Sample 4 ---
GROUND TRUTH: a giant panda rests peacefully on a rock.. tone : calm
PREDICTION:   a giant panda rests peacefully on a lush green enclosure......... tone : calm

--- Sample 5 ---
GROUND TRUTH: a fluffy, light brown bunny sits calmly in its cage.. tone : calm
PREDICTION:   a fluffy and white background, brightly in a dark surface........ tone : calm

--- Sample 6 ---
GROUND TRUTH: a large elephant walks through tall grass in the savanna.. tone : calm
PREDI

In [35]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
import nltk.translate.bleu_score as bleu

# Note: If you haven't done so, you must also ensure the NLTK Punkt tokenizer data is downloaded:
# import nltk
# nltk.download('punkt')
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model.pt'
try:
    new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
except FileNotFoundError:
    print(f"ERROR: Model file not found at {MODEL_SAVE_PATH}. Cannot run final test.")
    # raise # Re-add raise if you want the script to stop on file error

# --- 2. Inference (Text Generation) Function ---
@torch.no_grad()
# def generate_text_tf(model, eeg_signal, meta_signal, edge_index, edge_attr, max_len=64):
#     """Generates text from EEG and Metadata using greedy decoding."""
#     model.eval()
    
#     eeg_signal = eeg_signal.unsqueeze(0).to(device)
#     meta_signal = meta_signal.unsqueeze(0).to(device)
    
#     eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
#     meta_features = model.meta_encoder(meta_signal)
    
#     decoder_memory = eeg_features[:, 1:, :] 
#     memory_mask = None 

#     input_ids = torch.tensor([[SOS_ID]], dtype=torch.long, device=device)
#     generated_ids = []

#     for _ in range(max_len):
#         logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
#         next_token_logits = logits[:, -1, :] 
#         next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(0)
#         token_id = next_token_id.item()
        
#         if token_id == EOS_ID:
#             break
            
#         if token_id != PAD_ID:
#             generated_ids.append(token_id)
            
#         input_ids = torch.cat([input_ids, next_token_id.long()], dim=1)

#     predicted_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
#     return predicted_text


def generate_text_beam(model, eeg_signal, meta_signal, edge_index, edge_attr, 
                       beam_width=5, max_len=64):
    """
    Generates text using Beam Search for the Transformer Decoder.
    Note: Since the Transformer decoder processes the entire input sequence at once,
    this implementation feeds the growing input_ids to the decoder at each step.
    """
    model.eval()
    
    # Unsqueeze input to create a batch size of 1
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG and Metadata (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
    memory_mask = None 

    # 2. Initialization: Start with a list of beams
    # Each beam is: (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 3. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        # Iterate over all current beams
        for seq, score in beams:
            # Skip beams that are already finished
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            # Input sequence for the decoder (e.g., [SOS, w1, w2])
            input_ids = torch.tensor([seq], dtype=torch.long, device=device) # (1, current_len)

            # Decoder forward pass: Gets logits for all tokens up to current_len
            # Output logits is (1, current_len, vocab_size)
            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            
            # We only care about the logits for the last token in the sequence
            next_token_logits = logits[:, -1, :].squeeze(0) # (vocab_size)
            
            # Convert to log probabilities and find the top K (beam_width) candidates
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            # Create new candidate beams
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob # Accumulate log probability
                
                new_beams.append((new_seq, new_score))

        # Select the top 'beam_width' beams from the candidates
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Stop condition: If the top beam is finished
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 4. Final Output Selection
    best_seq = beams[0][0]
    
    # Remove SOS (pos 0) and EOS (pos -1, if present) tokens
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    return predicted_text
# --- 3. BLEU Evaluation Loop with Printing ---

# # Configuration
# NUM_SAMPLES_TO_PRINT = 10 
# samples_printed = 0
# bleu_scores = [] 

# print("\n--- Starting BLEU Evaluation and Qualitative Analysis ---")

# new_model.eval()
# test_progress_bar = tqdm(test_loader, desc="BLEU Evaluation", leave=True)

# start_time = time.time()
# global_sample_index = 0

# for eeg_b, meta_b, txt_b in test_progress_bar:
    
#     for i in range(eeg_b.shape[0]):
#         eeg_sample = eeg_b[i]
#         meta_sample = meta_b[i]
#         true_text_ids = txt_b[i]
        
#         # --- Generation ---
#         predicted_text = generate_text_tf(
#             new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
#         )
        
#         # --- Ground Truth Text ---
#         true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
#         if not true_text:
#             continue
            
#         # --- Tokenization for NLTK/BLEU ---
#         # Reference must be a list of lists (for single reference, it's [[tokens]])
#         reference_tokens = [word_tokenize(true_text)]
#         hypothesis_tokens = word_tokenize(predicted_text)

#         # --- Print Sample Analysis (Qualitative) ---
#         if samples_printed < NUM_SAMPLES_TO_PRINT:
#             print(f"\n--- Sample {global_sample_index + 1} ---")
#             print(f"GROUND TRUTH: {true_text}")
#             print(f"PREDICTION:   {predicted_text}")
#             samples_printed += 1

#         # --- Calculate BLEU (Cumulative BLEU-4) ---
#         try:
#             # Standard cumulative BLEU-4 weights: (1/4, 1/4, 1/4, 1/4)
#             score_bleu = bleu.sentence_bleu(reference_tokens, hypothesis_tokens, weights=(0.25, 0.25, 0.25, 0.25))
#             bleu_scores.append(score_bleu)
#         except ZeroDivisionError:
#             # Skip if the hypothesis is too short (less than 4 words)
#             pass

#         # Update progress bar
#         avg_bleu = statistics.mean(bleu_scores) if bleu_scores else 0
#         test_progress_bar.set_postfix(avg_BLEU=f"{avg_bleu:.4f}")
        
#         global_sample_index += 1

# # --- 4. Final Reporting ---
# end_time = time.time()
# formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

# final_bleu = statistics.mean(bleu_scores) if bleu_scores else 0.0

# print("\n=============================================")
# print(f"Test Set Evaluation Complete in {formatted_time}")
# print("=============================================")
# print(f"| Total Samples Evaluated:  {len(bleu_scores)} |")
# print("---------------------------------------------")
# print(f"| **Cumulative BLEU-4 Score:** {final_bleu:.4f} |")
# print("=============================================")


NUM_SAMPLES_TO_PRINT = 10 
samples_printed = 0
bleu_metric = evaluate.load('bleu')

predictions_list = []
references_list = []
global_sample_index = 0

print("\n--- Starting BLEU Evaluation (Corpus-level) using BEAM SEARCH ---")

new_model.eval()
test_progress_bar = tqdm(test_loader, desc="Beam Search Evaluation", leave=True)

start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i]
        true_text_ids = txt_b[i]
        
        # --- Generation using BEAM SEARCH ---
        predicted_text = generate_text_beam(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr, beam_width=5
        )
        
        # --- Ground Truth Text ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
        if not true_text:
            continue
            
        # --- Store for Corpus-Level Evaluation ---
        predictions_list.append(predicted_text)
        references_list.append([true_text]) # List of references (even if only one)

        # --- Print Sample Analysis (Qualitative) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            print(f"\n--- Sample {global_sample_index + 1} ---")
            print(f"GROUND TRUTH: {true_text}")
            print(f"PREDICTION:   {predicted_text}")
            samples_printed += 1
            
        global_sample_index += 1

# --- 4. Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Beam Search Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score (Beam 5):** {final_bleu_score:.4f} |")
print("=============================================")

Successfully loaded best model weights from: eeg-meta-text-spatiotemporal-transformer-model.pt

--- Starting BLEU Evaluation and Qualitative Analysis ---


[nltk_data] Downloading package punkt_tab to /home/poorna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


BLEU Evaluation:   0%|          | 0/88 [00:00<?, ?it/s]

/home/poorna/venvs/torch/lib64/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/poorna/venvs/torch/lib64/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/poorna/venvs/torch/lib64/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram or


--- Sample 1 ---
GROUND TRUTH: a school of orange fish swims around a vibrant coral reef.. tone : serene
PREDICTION:   a vibrant orange and white clownfish swim near a vibrant coral reef... tone : serene

--- Sample 2 ---
GROUND TRUTH: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : awe - inspiring
PREDICTION:   a powerful

--- Sample 3 ---
GROUND TRUTH: vibrant fireworks explode in the night sky.. tone : energetic
PREDICTION:   a vibrant fireworks explode in the night sky... tone : exciting

--- Sample 4 ---
GROUND TRUTH: a giant panda rests peacefully on a rock.. tone : calm
PREDICTION:   a giant panda rests peacefully on a lush green enclosure......... tone : calm

--- Sample 5 ---
GROUND TRUTH: a fluffy, light brown bunny sits calmly in its cage.. tone : calm
PREDICTION:   a fluffy and white background, brightly in a dark surface........ tone : calm

--- Sample 6 ---
GROUND TRUTH: a large elephant walks through tall grass in the savanna.. tone : calm
PREDI

In [24]:
import torch
import torch.nn.functional as F
import statistics
from tqdm.auto import tqdm
import time
import math
import evaluate 
import json # Used for object mapping

# --- NOTE: Define fallback dictionaries if not already global ---
try:
    # Attempt to use existing global inverse maps
    id_to_color = {v: k for k, v in globals()['color_to_id'].items()}
    id_to_scene = {v: k for k, v in globals()['scene_to_id'].items()}
    
    # Assuming object_mapping is loaded from file or defined elsewhere
    if 'object_mapping' not in globals():
        object_mapping = {}

except:
    # Fallback to simple ID printing if inverse maps are missing
    id_to_color = {}
    id_to_scene = {}
    object_mapping = {}
    print("Warning: Inverse metadata maps (id_to_color, etc.) not found. Printing raw IDs.")


# --- 1. Combined Inference Function (Text and Metadata) ---
@torch.no_grad()
def generate_text_and_meta(model, eeg_signal, meta_signal, edge_index, edge_attr, max_len=64, beam_width=5):
    """Generates text via Beam Search (or Greedy if beam_width=1) and extracts metadata predictions."""
    model.eval()
    
    # Unsqueeze input to create a batch size of 1
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device) # Metadata is used ONLY by the meta_encoder for conditioning
    
    # 1. Encode EEG and Metadata (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal) # This metadata is the conditioning input for the decoder
    
    decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
    memory_mask = None 

    # 2. Predict Metadata from CLS token (The Auxiliary Task)
    cls_token_feature = eeg_features[:, 0, :]
    meta_preds = model.meta_head(cls_token_feature).squeeze(0) # (Total_Meta_Classes)
    
    # Decode Metadata Predictions
    pred_color_id = meta_preds[:model.num_colors].argmax().item()
    pred_category_id = meta_preds[model.num_colors:model.num_colors + model.num_categories].argmax().item()
    pred_object_logits = meta_preds[model.num_colors + model.num_categories:]
    
    # Simple thresholding (argmax is sufficient for the single predicted object name)
    pred_object_id = pred_object_logits.argmax().item()


    # 3. Text Generation (Greedy Search Logic)
    initial_seq = [SOS_ID]
    generated_ids = []
    
    input_ids = torch.tensor([initial_seq], dtype=torch.long, device=device) # (1, 1)

    for _ in range(max_len):
        logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
        next_token_logits = logits[:, -1, :].squeeze(0)
        
        # Greedy choice (argmax)
        next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(0)
        token_id = next_token_id.item()
        
        if token_id == EOS_ID: break
        if token_id != PAD_ID: generated_ids.append(token_id)
            
        input_ids = torch.cat([input_ids, next_token_id.long()], dim=1)

    predicted_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # 4. Return
    return (predicted_text, pred_color_id, pred_category_id, pred_object_id)


# --- 2. BLEU Evaluation Loop (Modified for Metadata Printing) ---

# Configuration
NUM_SAMPLES_TO_PRINT = 10 
samples_printed = 0
bleu_metric = evaluate.load('bleu')

predictions_list = []
references_list = []
global_sample_index = 0

print("\n--- Starting Combined Evaluation (Text & Metadata Prediction) ---")

new_model.eval()
test_progress_bar = tqdm(test_loader, desc="Combined Evaluation", leave=True)

start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i] # This is the TRUE metadata for comparison
        true_text_ids = txt_b[i]
        
        # --- Generation & Prediction ---
        predicted_text, pred_color_id, pred_category_id, pred_object_id = generate_text_and_meta(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
        )
        
        # --- Ground Truth Decoding ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
        # Extract TRUE Metadata (for printing comparison)
        true_color_id = int(meta_sample[0].item())
        true_category_id = int(meta_sample[1].item())
        
        if not true_text:
            continue
            
        # --- Store for Corpus-Level Evaluation ---
        predictions_list.append(predicted_text)
        references_list.append([true_text])

        # --- Print Sample Analysis (Qualitative & Metadata Check) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            
            # Decode names using the maps (falls back to ID if not found)
            pred_color_name = id_to_color.get(pred_color_id, f"ID: {pred_color_id}")
            true_color_name = id_to_color.get(true_color_id, f"ID: {true_color_id}")
            pred_category_name = id_to_scene.get(pred_category_id, f"ID: {pred_category_id}")
            true_category_name = id_to_scene.get(true_category_id, f"ID: {true_category_id}")
            pred_object_name = object_mapping.get(str(pred_object_id), f"ID: {pred_object_id}")
            
            # Find the TRUE object name(s) for comparison
            true_object_indices = meta_sample[2:].nonzero(as_tuple=True)[0]
            true_object_names = [object_mapping.get(str(idx.item()), f"ID: {idx.item()}") for idx in true_object_indices]
            if not true_object_names: true_object_names = ["None"]


            print(f"\n--- Sample {global_sample_index + 1} ---")
            print(f"GROUND TRUTH TEXT: {true_text}")
            print(f"PREDICTED TEXT:    {predicted_text}")
            print("\n  --- METADATA PREDICTION (from EEG CLS) ---")
            print(f"  Color:      Truth='{true_color_name}' | Pred='{pred_color_name}'")
            print(f"  Category:   Truth='{true_category_name}' | Pred='{pred_category_name}'")
            print(f"  Object(s):  Truth={', '.join(true_object_names)} | Pred='{pred_object_name}'")
            samples_printed += 1
            
        global_sample_index += 1

# --- 3. Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score:** {final_bleu_score:.4f} |")
print("=============================================")


--- Starting Combined Evaluation (Text & Metadata Prediction) ---


NameError: name 'new_model' is not defined

In [ ]:
import torch
import torch.nn.functional as F
import statistics
from tqdm.auto import tqdm
import time
import math
import evaluate 
import json # Used for object mapping

# --- NOTE: Define fallback dictionaries if not already global ---
try:
    # Attempt to use existing global inverse maps
    id_to_color = {v: k for k, v in globals()['color_to_id'].items()}
    id_to_scene = {v: k for k, v in globals()['scene_to_id'].items()}
    
    # Assuming object_mapping is loaded from file or defined elsewhere
    if 'object_mapping' not in globals():
        object_mapping = {}

except:
    # Fallback to simple ID printing if inverse maps are missing
    id_to_color = {}
    id_to_scene = {}
    object_mapping = {}
    print("Warning: Inverse metadata maps (id_to_color, etc.) not found. Printing raw IDs.")


# --- 1. Combined Inference Function (Text and Metadata) ---
@torch.no_grad()
def generate_text_and_meta(model, eeg_signal, meta_signal, edge_index, edge_attr, max_len=64, beam_width=1):
    """Generates text via Beam Search (or Greedy if beam_width=1) and extracts metadata predictions."""
    model.eval()
    
    # Unsqueeze input to create a batch size of 1
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device) # Metadata is used ONLY by the meta_encoder for conditioning
    
    # 1. Encode EEG and Metadata (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal) # This metadata is the conditioning input for the decoder
    
    decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
    memory_mask = None 

    # 2. Predict Metadata from CLS token (The Auxiliary Task)
    cls_token_feature = eeg_features[:, 0, :]
    meta_preds = model.meta_head(cls_token_feature).squeeze(0) # (Total_Meta_Classes)
    
    # Decode Metadata Predictions
    pred_color_id = meta_preds[:model.num_colors].argmax().item()
    pred_category_id = meta_preds[model.num_colors:model.num_colors + model.num_categories].argmax().item()
    pred_object_logits = meta_preds[model.num_colors + model.num_categories:]
    
    # Simple thresholding (argmax is sufficient for the single predicted object name)
    pred_object_id = pred_object_logits.argmax().item()


    # 3. Text Generation (Greedy Search Logic)
    initial_seq = [SOS_ID]
    generated_ids = []
    
    input_ids = torch.tensor([initial_seq], dtype=torch.long, device=device) # (1, 1)

    for _ in range(max_len):
        logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
        next_token_logits = logits[:, -1, :].squeeze(0)
        
        # Greedy choice (argmax)
        next_token_id = next_token_logits.argmax(dim=-1).unsqueeze(0)
        token_id = next_token_id.item()
        
        if token_id == EOS_ID: break
        if token_id != PAD_ID: generated_ids.append(token_id)
            
        input_ids = torch.cat([input_ids, next_token_id.long()], dim=1)

    predicted_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # 4. Return
    return (predicted_text, pred_color_id, pred_category_id, pred_object_id)


# --- 2. BLEU Evaluation Loop (Modified for Metadata Printing) ---

# Configuration
NUM_SAMPLES_TO_PRINT = 10 
samples_printed = 0
bleu_metric = evaluate.load('bleu')

predictions_list = []
references_list = []
global_sample_index = 0

print("\n--- Starting Combined Evaluation (Text & Metadata Prediction) ---")

new_model.eval()
test_progress_bar = tqdm(test_loader, desc="Combined Evaluation", leave=True)

start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i] # This is the TRUE metadata for comparison
        true_text_ids = txt_b[i]
        
        # --- Generation & Prediction ---
        predicted_text, pred_color_id, pred_category_id, pred_object_id = generate_text_and_meta(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
        )
        
        # --- Ground Truth Decoding ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
        # Extract TRUE Metadata (for printing comparison)
        true_color_id = int(meta_sample[0].item())
        true_category_id = int(meta_sample[1].item())
        
        if not true_text:
            continue
            
        # --- Store for Corpus-Level Evaluation ---
        predictions_list.append(predicted_text)
        references_list.append([true_text])

        # --- Print Sample Analysis (Qualitative & Metadata Check) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            
            # Decode names using the maps (falls back to ID if not found)
            pred_color_name = id_to_color.get(pred_color_id, f"ID: {pred_color_id}")
            true_color_name = id_to_color.get(true_color_id, f"ID: {true_color_id}")
            pred_category_name = id_to_scene.get(pred_category_id, f"ID: {pred_category_id}")
            true_category_name = id_to_scene.get(true_category_id, f"ID: {true_category_id}")
            pred_object_name = object_mapping.get(str(pred_object_id), f"ID: {pred_object_id}")
            
            # Find the TRUE object name(s) for comparison
            true_object_indices = meta_sample[2:].nonzero(as_tuple=True)[0]
            true_object_names = [object_mapping.get(str(idx.item()), f"ID: {idx.item()}") for idx in true_object_indices]
            if not true_object_names: true_object_names = ["None"]


            print(f"\n--- Sample {global_sample_index + 1} ---")
            print(f"GROUND TRUTH TEXT: {true_text}")
            print(f"PREDICTED TEXT:    {predicted_text}")
            print("\n  --- METADATA PREDICTION (from EEG CLS) ---")
            print(f"  Color:      Truth='{true_color_name}' | Pred='{pred_color_name}'")
            print(f"  Category:   Truth='{true_category_name}' | Pred='{pred_category_name}'")
            print(f"  Object(s):  Truth={', '.join(true_object_names)} | Pred='{pred_object_name}'")
            samples_printed += 1
            
        global_sample_index += 1

# --- 3. Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score:** {final_bleu_score:.4f} |")
print("=============================================")